┌─────────────────────────────────────────────────────────┐
│                    DATA LAYER                           │
│  (Your existing pipeline — historical + live streaming) │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  FEATURE LAYER                          │
│  (Alignment, resampling, all engineered features)       │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  SIGNAL LAYER                           │
│  (Individual setups A/B/C/D — each independently)      │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  RISK LAYER                             │
│  (Position sizing, max drawdown, correlation limits)    │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│               EXECUTION LAYER                           │
│  (Order routing, slippage, Binance API)                 │
└─────────────────────────────────────────────────────────┘

In [ ]:
from orderflow import futures_agg_trades, get_oi, get_mark_price_klines, spot_agg_trades, get_premium_index_klines
import pandas as pd
import numpy as np

symbol = "BTCUSDT"
start = pd.Timestamp("2026-06-09 12:00")
end   = pd.Timestamp("2026-06-10 18:00")
timeframe = "5min" 

dates = [d.strftime("%Y-%m-%d")
         for d in pd.date_range((start - pd.Timedelta(days=1)).normalize(), end.normalize(), freq="D")]
both = lambda fn: pd.concat([fn(symbol, d) for d in dates]).sort_index()

fut_trades = both(futures_agg_trades)
oi         = both(get_oi)
context    = both(get_mark_price_klines)
spot       = both(spot_agg_trades)
premium    = both(get_premium_index_klines)

ohlc = context.resample(timeframe).agg({"open": "first", "high": "max", "low": "min", "close": "last"})

fut_cvd  = fut_trades["volume_delta"].resample(timeframe).sum().cumsum()
spot_cvd = spot["volume_delta"].resample(timeframe).sum().cumsum()

I = 0.0001
step = premium.index.to_series().diff().median()
n = int(pd.Timedelta("8h") / step)
w = np.arange(1, n + 1)
p_avg = premium["close"].rolling(n).apply(lambda x: x @ w / w.sum(), raw=True)
funding_rate = p_avg + (I - p_avg).clip(-0.0005, 0.0005)
funding_annualized = (funding_rate * 3 * 365).resample(timeframe).last()

oi_tf = oi["sum_open_interest"].resample(timeframe).last()

In [ ]:
combined = pd.concat([
    fut_cvd.rename("fut_cumulative_volume_delta"),
    spot_cvd.rename("spot_cumulative_volume_delta"),
    ohlc,
    oi_tf.rename("sum_open_interest"),
    funding_annualized.rename("funding_rate_annualized"),
], axis=1)

combined = combined.loc[start.floor(timeframe):end]

for col in ["fut_cumulative_volume_delta", "spot_cumulative_volume_delta"]:
    combined[col] = combined[col] - combined[col].dropna().iloc[0]

combined

,fut_cumulative_volume_delta,spot_cumulative_volume_delta,open,high,low,close,sum_open_interest,funding_rate_annualized
2026-06-09 12:00:00,0.000,0.00000,62682.590014,62830.624681,62663.761732,62789.375290,97637.256,-0.002440
2026-06-09 12:05:00,141.504,18.95204,62787.500000,62870.966259,62733.104739,62783.100000,97493.947,-0.004115
2026-06-09 12:10:00,114.397,21.92118,62783.000000,62802.141515,62697.100000,62802.141515,97476.878,-0.004457
2026-06-09 12:15:00,67.316,16.21623,62803.400000,62814.600000,62713.815246,62764.738435,97444.664,-0.004350
2026-06-09 12:20:00,10.721,17.59789,62760.941261,62760.941261,62632.500000,62711.500000,97418.302,-0.005060
...,...,...,...,...,...,...,...,...
2026-06-10 17:40:00,-3861.397,-1440.53007,61740.383232,61818.700000,61623.328630,61720.255427,99866.481,-0.003091
2026-06-10 17:45:00,-3735.571,-1415.95396,61714.145486,62015.700000,61693.889580,62015.700000,100495.303,0.001481
2026-06-10 17:50:00,-3897.787,-1443.20197,62015.800000,62015.800000,61810.354486,61863.800000,100389.405,0.000225
2026-06-10 17:55:00,-3822.382,-1441.85832,61863.700000,61977.200000,61863.700000,61944.400000,100363.108,0.000071


In [3]:
from orderflow import build_order_flow_chart

my_layout = [
    {"type": "candlestick", "title": "Price Action", "name": "Candlestick", "y_title": "Price (USDT)"},
    {"type": "line", "title": "Open Interest", "column": "sum_open_interest", "name": "OI Coins", "color": "purple", "y_title": "OI"},
    {"type": "delta_bars", "title": "Funding Rate", "column": "funding_rate_annualized", "name": "Funding", "color": "orange", "y_title": "Rate"},
    {"type": "delta_bars", "title": "Spot CVD", "column": "spot_cumulative_volume_delta", "name": "Spot Delta", "y_title": "Delta"},
    {"type": "delta_bars", "title": "Futures CVD", "column": "fut_cumulative_volume_delta", "name": "Futures Delta", "y_title": "Delta"}
]

fig = build_order_flow_chart(combined, f"{start:%Y-%m-%d %H:%M} → {end:%Y-%m-%d %H:%M}", config=my_layout, timeframe=timeframe)
fig.show()